# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the **FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step pulls the schema and associated metadata from the provided URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity (record set, field, column) will be referenced by its `@id` as per Croissant schema.

In [ ]:
# List available record sets and their `@id`s
record_sets_info = dataset.metadata.to_json().get('recordSet', [])
if not record_sets_info:
    print("No record sets found in schema.")
else:
    print("Available Record Sets:")
    for rs in record_sets_info:
        print(f" - {rs.get('@id', '[No ID]')}: {rs.get('name', '[No Name]')}")
    # For demonstration, select the first record set
    chosen_record_set_id = record_sets_info[0]['@id']
    print(f"\nUsing record set: {chosen_record_set_id}")

    # List available fields and their @id
    fields_info = record_sets_info[0].get('field', [])
    print("Fields in this record set:")
    for f in fields_info:
        print(f" - {f.get('@id', '[No ID]')}: {f.get('name', '[No Name]')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

This step creates a DataFrame for each selected record set.

In [ ]:
dataframes = {}

# Make a list of all recordSet @ids
record_sets_info = dataset.metadata.to_json().get('recordSet', [])
record_set_ids = [rs['@id'] for rs in record_sets_info]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if record_set_ids:
    print("DataFrame columns for first record set:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes.

Fields, columns, or attributes are always referenced by their `@id` or mapped label.

In [ ]:
# Choose a record set and numeric field for analysis
record_sets_info = dataset.metadata.to_json().get('recordSet', [])

# Demonstrate using the first record set and first numeric field
if record_sets_info:
    chosen_record_set_id = record_sets_info[0]['@id']
    df = dataframes[chosen_record_set_id]
    fields = record_sets_info[0].get('field', [])
    # Identify a numeric field (Look for type 'Integer'/'Float')
    numeric_field_id = None
    group_field_id = None
    for f in fields:
        if f.get('dataType') in ['schema:Integer', 'schema:Float']:
            numeric_field_id = f['@id']
            break
    # Choose a field for grouping (e.g., demographic string field)
    for f in fields:
        if f.get('dataType') == 'schema:Text':
            group_field_id = f['@id']
            break

    if numeric_field_id and numeric_field_id in df.columns:
        # Set threshold for EDA filter
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped {numeric_field_id} by {group_field_id} (mean values):")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between key fields using standard plotting libraries. This helps to intuitively understand trends or outliers in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Example visualization: distribution of the numeric field
if record_sets_info and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If there is a group field, visualize mean values by group
    if group_field_id and group_field_id in df.columns:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        grouped.plot(kind='bar', figsize=(10,6))
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, review, and analyze the **FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the `mlcroissant` library. Key findings include:

- The dataset is rich in survey responses with detailed demographic and mental health indicators.
- Fields are referenced throughout by their `@id`, ensuring precise tracing for reproducible analysis.
- Simple EDA and visualizations reveal distributions and potential subgroup trends in numeric responses.

Further analysis can focus on deeper relationships, correlation with demographic variables, and models for mental health outcome prediction.